# LoanSphere – Multi-Agent Loan Underwriting Demo

**CSE476 | Agentic AI Project**

This notebook demonstrates the existing LoanSphere implementation. It creates applicant records, sends them to the `UnderwritingManager`, and displays the decision and multi-agent execution trace.

**Run this notebook from the project root** (`Loan-Underwriting-System`) using the project's `.venv` Python kernel. No agent or tool files need to be changed.

In [2]:
%pip install semantic-kernel==1.46.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.1.53 requires packaging<24.0,>=23.2, but you have packaging 26.3 which is incompatible.
You should consider upgrading via the 'C:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.



  Using cached semantic_kernel-1.46.0-py3-none-any.whl (949 kB)
  Using cached aiortc-1.15.0-py3-none-any.whl (93 kB)
  Using cached azure_identity-1.27.0-py3-none-any.whl (203 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl (60 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl (141 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl (69 kB)
  Using cached azure_ai_projects-2.4.0-py3-none-any.whl (357 kB)
  Using cached prance-26.7.19.0-py3-none-any.whl (2.8 MB)
  Using cached openapi_core-0.19.5-py3-none-any.whl (106 kB)
  Using cached mcp-1.30.0-py3-none-any.whl (234 kB)
  Using cached pybars4-0.9.13.tar.gz (29 kB)
  Using cached cloudevents-2.2.0-py3-none-any.whl (98 kB)
  Using cached azure_ai_agents-1.2.0b6-py3-none-any.whl (217 kB)
  Using cached pyopenssl-26.4.0-py3-none-any.whl (56 kB)
  Using cached aioice-0.10.2-py3-none-any.whl (24 kB)
  Using cached pylibsrtp-1.0.0-cp310-abi3-win_amd64.whl (1.6 MB)
  Using cached pyee-13.0.1-py3-none-any.wh

In [3]:
from memory.applicant_record import ApplicantRecord
from agents.underwriting_manager import UnderwritingManager

print('LoanSphere modules imported successfully.')

LoanSphere modules imported successfully.


## 1. Import the existing project modules

The notebook reuses the same `ApplicantRecord` and `UnderwritingManager` classes as the Streamlit frontend.

## 2. Define sample loan applications

The examples use the synthetic credit records already stored in `memory/credit_records.json`:

- `APP001`: clear credit history.
- `APP002`: credit risk due to defaults and missed payments.
- `TEST003`: clear credit history, but the selected income and loan details fail affordability.
- `NEW004`: no income supplied; the manager should refer the application before calculating EMI or affordability.

The same loan terms are used where applicable: ₹5,00,000 principal, 36 months, and 10% annual interest.

In [4]:
sample_applications = [
    {
        'applicant_id': 'APP001',
        'application': {
            'monthly_income': 70000,
            'existing_monthly_emi': 5000,
            'loan_amount': 500000,
            'tenure_months': 36,
            'annual_interest_rate': 10
        },
        'expected_decision': 'SANCTION'
    },
    {
        'applicant_id': 'TEST003',
        'application': {
            'monthly_income': 50000,
            'existing_monthly_emi': 5000,
            'loan_amount': 500000,
            'tenure_months': 36,
            'annual_interest_rate': 10
        },
        'expected_decision': 'DECLINE'
    },
    {
        'applicant_id': 'APP002',
        'application': {
            'monthly_income': 70000,
            'existing_monthly_emi': 5000,
            'loan_amount': 500000,
            'tenure_months': 36,
            'annual_interest_rate': 10
        },
        'expected_decision': 'REFER'
    },
    {
        'applicant_id': 'NEW004',
        'application': {
            'existing_monthly_emi': 5000,
            'loan_amount': 500000,
            'tenure_months': 36,
            'annual_interest_rate': 10
        },
        'expected_decision': 'REFER'
    }
]

print(f'{len(sample_applications)} sample applications are ready.')

4 sample applications are ready.


## 3. Run the multi-agent underwriting workflow

For each application, the Underwriting Manager invokes the Document Agent first. If the required information is valid, it proceeds with EMI, affordability, and credit-history assessment. The manager then validates the specialist results and records the final decision.

The manager also appends each execution to `logs/run_log.jsonl`, just as it does when the Streamlit application is used.

In [5]:
results = []

for sample in sample_applications:
    print('\n' + '=' * 75)
    print(f"PROCESSING APPLICANT: {sample['applicant_id']}")
    print('=' * 75)

    applicant = ApplicantRecord(
        applicant_id=sample['applicant_id'],
        application=sample['application']
    )

    manager = UnderwritingManager()
    assessed_applicant = manager.process_application(applicant)
    results.append({
        'applicant': assessed_applicant,
        'expected_decision': sample['expected_decision']
    })


PROCESSING APPLICANT: APP001

========== UNDERWRITING MANAGER ==========

[Manager] Delegating to Document Agent...

[Document Agent] Processing application...
Extracted details: {'net_monthly_income': 70000, 'existing_monthly_emi': 5000, 'requested_loan_amount': 500000, 'tenure_months': 36, 'annual_interest_rate': 10}
Validation result: {'valid': True, 'missing_fields': [], 'message': 'Application details are valid.'}

[Manager] Delegating to EMI Computation Agent...

[EMI Agent] Calculating loan EMI...
Monthly EMI: 16133.59
Total repayment: 580809.24
Total interest: 80809.24

[Manager] Delegating to Affordability Agent...

[Affordability Agent] Evaluating loan affordability...
Total monthly obligations: ₹21133.59
Obligation ratio: 30.19%
Verdict: AFFORDABLE

[Manager] Delegating to Credit-History Agent...

[Credit-History Agent] Checking credit history...
Credit status: CLEAR
- No major credit red flags found.

[Manager] Validating specialist results...

[Manager] Evaluating special

## 4. Display the decisions and financial results

This cell summarizes the result returned by the Underwriting Manager for each applicant.

In [6]:
for item in results:
    applicant = item['applicant']
    print('\n' + '-' * 65)
    print(f'Applicant ID: {applicant.applicant_id}')
    print(f"Decision: {applicant.decision}")
    print(f"Expected decision: {item['expected_decision']}")
    print(f"Decision check: {'PASS' if applicant.decision == item['expected_decision'] else 'CHECK'}")
    print(f"Monthly EMI: {applicant.computed_emi if applicant.computed_emi is not None else 'Not calculated'}")
    print(f"Total monthly obligations: {applicant.total_monthly_obligations if applicant.total_monthly_obligations is not None else 'Not calculated'}")
    print(f"Affordability: {applicant.affordability_verdict or 'Not evaluated'}")
    print(f"Credit status: {applicant.credit_status or 'Not evaluated'}")
    print(f"Justification: {applicant.justification}")


-----------------------------------------------------------------
Applicant ID: APP001
Decision: SANCTION
Expected decision: SANCTION
Decision check: PASS
Monthly EMI: 16133.59
Total monthly obligations: 21133.59
Affordability: AFFORDABLE
Credit status: CLEAR
Justification: Application satisfies document verification, affordability requirements, and credit-history checks.

-----------------------------------------------------------------
Applicant ID: TEST003
Decision: DECLINE
Expected decision: DECLINE
Decision check: PASS
Monthly EMI: 16133.59
Total monthly obligations: 21133.59
Affordability: NOT AFFORDABLE
Credit status: CLEAR
Justification: Total monthly obligations are 42.27% of net monthly income, which exceeds the permitted 40% threshold.

-----------------------------------------------------------------
Applicant ID: APP002
Decision: REFER
Expected decision: REFER
Decision check: PASS
Monthly EMI: 16133.59
Total monthly obligations: 21133.59
Affordability: AFFORDABLE
Credit s

## 5. Display the multi-agent execution trace

The trace shows the sequence of actions performed during each application assessment. It provides a simple way to inspect how the manager and specialist agents contributed to the final result.

In [7]:
for item in results:
    applicant = item['applicant']
    print(f'\nEXECUTION TRACE — {applicant.applicant_id}')
    print('-' * 65)

    for step_number, trace_message in enumerate(applicant.trace, start=1):
        print(f'{step_number}. {trace_message}')


EXECUTION TRACE — APP001
-----------------------------------------------------------------
1. Underwriting Manager received application.
2. Document Agent: Extracted application details.
3. Document Agent: Validated application details.
4. EMI Agent: Calculated monthly EMI.
5. EMI Agent: Calculated total repayment and interest.
6. Affordability Agent calculated monthly obligation ratio.
7. Affordability verdict: AFFORDABLE
8. Credit-History Agent retrieved the applicant's credit record.
9. Credit status: CLEAR
10. Underwriting Manager made final decision: SANCTION
11. Decision record created: SANCTION
12. Final decision generated.

EXECUTION TRACE — TEST003
-----------------------------------------------------------------
1. Underwriting Manager received application.
2. Document Agent: Extracted application details.
3. Document Agent: Validated application details.
4. EMI Agent: Calculated monthly EMI.
5. EMI Agent: Calculated total repayment and interest.
6. Affordability Agent calcu

## 6. Verify the scenario outcomes

The final check compares the decisions generated by the manager with the expected outcomes for the four demonstration cases.

In [8]:
all_passed = True

for item in results:
    applicant = item['applicant']
    expected = item['expected_decision']
    passed = applicant.decision == expected
    all_passed = all_passed and passed
    print(f'{applicant.applicant_id}: expected={expected}, actual={applicant.decision}, result={"PASS" if passed else "FAIL"}')

print('\n' + ('All demonstration scenarios passed.' if all_passed else 'One or more scenarios need investigation.'))
assert all_passed, 'A demonstration scenario did not produce its expected decision.'

APP001: expected=SANCTION, actual=SANCTION, result=PASS
TEST003: expected=DECLINE, actual=DECLINE, result=PASS
APP002: expected=REFER, actual=REFER, result=PASS
NEW004: expected=REFER, actual=REFER, result=PASS

All demonstration scenarios passed.


## 7. Conclusion

This notebook demonstrates the end-to-end multi-agent loan underwriting workflow. The Document Agent validates the application, the EMI Agent performs deterministic repayment calculations, the Affordability Agent checks the income threshold, and the Credit-History Agent evaluates the stored credit record. The Underwriting Manager combines the results and produces a final SANCTION, DECLINE, or REFER decision.

The notebook reuses the same project modules and execution log as the Streamlit frontend. The credit-history examples are synthetic and intended only for academic demonstration.